<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_05_04_LightGBM_one2one.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_05_04 - ONE2ONE - LightGBM**

## **Introducción**

**LightGBM** es un modelo de **gradient boosting sobre árboles** diseñado para ser más rápido y eficiente que implementaciones tradicionales.

Al igual que XGBoost:

* entrena árboles **secuencialmente**
* cada árbol intenta corregir el error del modelo anterior

Pero LightGBM se caracteriza por:

* mayor velocidad en datasets grandes
* menor consumo de memoria
* buen rendimiento en datos tabulares
* soporte nativo de **early stopping**

**Por qué usarlo en tu proyecto**

Después de probar:

* **Ridge** → sin señal lineal útil
* **Random Forest** → baseline no lineal
* **XGBoost** → mejora limitada y sobreajuste temprano

LightGBM es un siguiente paso natural porque:

* sigue siendo muy fuerte en tabular
* puede capturar no linealidades e interacciones
* suele entrenar más rápido que XGBoost
* permite monitorear fácilmente train/valid

# **Bloque común**

## **1. Imports + paths**

In [41]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [42]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **3. Rutas de variables X e y, y scalers**

In [43]:
from pathlib import Path
import os
import pandas as pd
import joblib


XY_DELTA_DIR = DRIVE_DIR / Path(
    os.environ.get("XY_DELTA_DIR", "data/splits/")
)

XY_DELTA_DIR_SCALED = DRIVE_DIR / Path(
    os.environ.get("XY_DELTA_DIR_SCALED", "data/scaled/")
)

SCALERS_DIR = DRIVE_DIR / Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

TARGETS = ["delta_60", "delta_90"]
SPLITS = ["train", "valid", "test"]


def load_mnq_tabular_split(
    target: str,
    split: str,
    scaled: bool = False,
    return_scaler: bool = False,
):
    if target not in TARGETS:
        raise ValueError(f"target inválido: {target}. Esperados: {TARGETS}")

    if split not in SPLITS:
        raise ValueError(f"split inválido: {split}. Esperados: {SPLITS}")

    x_path = (
        XY_DELTA_DIR_SCALED / f"mnq_{target}_X_{split}_scaled.parquet"
        if scaled
        else XY_DELTA_DIR / f"mnq_{target}_X_{split}.parquet"
    )
    y_path = XY_DELTA_DIR / f"mnq_{target}_y_{split}.parquet"

    if not x_path.exists():
        raise FileNotFoundError(f"No existe X: {x_path}")
    if not y_path.exists():
        raise FileNotFoundError(f"No existe y: {y_path}")

    X = pd.read_parquet(x_path)
    y = pd.read_parquet(y_path)

    if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
        y = y.iloc[:, 0]

    if not return_scaler:
        return X, y

    scaler = None
    if scaled:
        scaler_path = SCALERS_DIR / f"scaler_{target}.pkl"
        if scaler_path.exists():
            scaler = joblib.load(scaler_path)

    return X, y, scaler

In [44]:
#Sin escalado
SIN_ESCALADO = '''
X_train, y_train = load_mnq_tabular_split(
    target="delta_60",
    split="train",
    scaled=False,
)
'''

In [45]:
#Escalado
ESCALADO = '''
X_train, y_train = load_mnq_tabular_split(
    target="delta_60",
    split="train",
    scaled=True,
)
'''


ESCALADO_ESCALADOR = '''
X_train, y_train, scaler = load_mnq_tabular_split(
    target="delta_60",
    split="train",
    scaled=True,
    load_scaler=True,
)
'''

In [46]:
import pandas as pd
import numpy as np


def validate_tabular_dataset(
    X: pd.DataFrame,
    y: pd.Series | pd.DataFrame,
    *,
    name: str = "",
    check_index_alignment: bool = True,
    check_sorted: bool = True,
    date_col: str | None = None,
    verbose: bool = True,
):
    """
    Valida consistencia de un dataset tabular (X, y).

    Checks:
    - shapes
    - NaNs / inf
    - alineación de índices
    - orden temporal (opcional)
    - duplicados

    Retorna
    -------
    dict con flags de validación
    """

    report = {}

    # -------- Convertir y --------
    if isinstance(y, pd.DataFrame) and y.shape[1] == 1:
        y = y.iloc[:, 0]

    # -------- Shapes --------
    report["n_samples_X"] = X.shape[0]
    report["n_samples_y"] = y.shape[0]
    report["n_features"] = X.shape[1]
    report["shape_match"] = X.shape[0] == y.shape[0]

    # -------- NaNs / inf --------
    report["X_has_nan"] = X.isna().any().any()
    report["y_has_nan"] = y.isna().any()

    report["X_has_inf"] = np.isinf(X.select_dtypes(include=[np.number])).any().any()
    report["y_has_inf"] = np.isinf(y).any()

    # -------- Índices --------
    if check_index_alignment:
        report["index_equal"] = X.index.equals(y.index)
    else:
        report["index_equal"] = None

    # -------- Orden temporal --------
    if check_sorted:
        if date_col and date_col in X.columns:
            report["sorted_by_date"] = X[date_col].is_monotonic_increasing
        else:
            report["sorted_by_index"] = X.index.is_monotonic_increasing
    else:
        report["sorted"] = None

    # -------- Duplicados --------
    report["duplicate_index"] = X.index.duplicated().any()

    # -------- Print --------
    if verbose:
        print(f"\n=== VALIDATION: {name} ===")
        for k, v in report.items():
            print(f"{k}: {v}")

        if not report["shape_match"]:
            print("⚠️ ERROR: X e y no tienen mismo número de filas")

        if report["X_has_nan"] or report["y_has_nan"]:
            print("⚠️ WARNING: Hay NaNs")

        if report["X_has_inf"] or report["y_has_inf"]:
            print("⚠️ WARNING: Hay valores infinitos")

        if check_index_alignment and not report["index_equal"]:
            print("⚠️ WARNING: Índices no alineados")

    return report

## **4. Reproducibilidad**

In [47]:
def set_seeds(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [48]:
from pathlib import Path
import os
import sys
import importlib

DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

from metrics.one2one_metrics import evaluate_regression_predictions, print_metrics

print("OK - imports metrics.*")

import metrics.one2one_metrics as m

#print(m.__doc__)
#print(m.evaluate_regression_predictions.__doc__)


OK - imports metrics.*


## **6. Métricas Machine Learning**

In [49]:
import pandas as pd


def evaluate_model_on_split(
    model,
    X,
    y,
    *,
    split_name: str,
    model_name: str,
    target_name: str,
):
    """
    Evalúa un modelo sobre un split dado y devuelve:
    - y_pred
    - dict de métricas
    """
    y_pred = model.predict(X)

    metrics = evaluate_regression_predictions(
        y_true=y,
        y_pred=y_pred,
        split_name=split_name,
        model_name=model_name,
        target_name=target_name,
    )

    return y_pred, metrics


def metrics_to_df(metrics: dict) -> pd.DataFrame:
    """
    Convierte un dict de métricas en una fila de DataFrame.
    Versión final sin redundancias (ni window_size ni horizon).
    """
    row = {
        "model": metrics.get("model"),
        "split": metrics.get("split"),
        "target": metrics.get("target"),
        "n_samples": metrics.get("n_samples"),
        "mae": metrics.get("mae"),
        "rmse": metrics.get("rmse"),
        "r2": metrics.get("r2"),
        "directional_accuracy": metrics.get("directional_accuracy"),
    }

    return pd.DataFrame([row])

def evaluate_model_on_bundle(
    model,
    bundle: dict,
    *,
    model_name: str,
    target_name: str,
    window_size: int | None = None,
    horizon: int | None = None,
    splits: tuple[str, ...] = ("valid", "test"),
):
    """
    Evalúa un modelo en varios splits de un bundle.

    Estructura esperada de bundle:
    bundle = {
        "train": {"X": ..., "y": ...},
        "valid": {"X": ..., "y": ...},
        "test":  {"X": ..., "y": ...},
    }

    Retorna
    -------
    predictions : dict
        Predicciones por split.
    metrics_dict : dict
        Métricas por split.
    metrics_df : pd.DataFrame
        Tabla consolidada.
    """
    predictions = {}
    metrics_dict = {}
    frames = []

    for split in splits:
        X = bundle[split]["X"]
        y = bundle[split]["y"]

        y_pred, metrics = evaluate_model_on_split(
            model=model,
            X=X,
            y=y,
            split_name=split,
            model_name=model_name,
            target_name=target_name,
        )

        predictions[split] = y_pred
        metrics_dict[split] = metrics
        frames.append(
            metrics_to_df(
                metrics,
                window_size=window_size,
                horizon=horizon,
            )
        )

    metrics_df = pd.concat(frames, ignore_index=True)

    return predictions, metrics_dict, metrics_df

## **7. Gestión de dataset de métricas**

In [50]:
def load_one2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/one2one_metrics",
) -> pd.DataFrame:
    path = Path(base_dir) / f"one2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [51]:
from pathlib import Path
import pandas as pd

def save_one2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/one2one_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"one2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

# **DEFINICIÓN DE MODELO**

## **8. Definición del modelo — placeholder**

### **8.1. Modelo Ridge Regression (one2one)**

### **8.2. Carga de X e y**

In [52]:
TARGETS = ["delta_60", "delta_90"]

data = {}

for target in TARGETS:
    print(f"\n==============================")
    print(f"CARGANDO DATASET: {target}")
    print(f"==============================")

    # -------- TRAIN --------
    X_train, y_train = load_mnq_tabular_split(
        target=target,
        split="train",
        scaled=False,   # ✔ correcto
    )

    validate_tabular_dataset(
        X_train,
        y_train,
        name=f"train_{target}",
    )

    # -------- VALID --------
    X_valid, y_valid = load_mnq_tabular_split(
        target=target,
        split="valid",
        scaled=False,
    )

    validate_tabular_dataset(
        X_valid,
        y_valid,
        name=f"valid_{target}",
    )

    # -------- TEST --------
    X_test, y_test = load_mnq_tabular_split(
        target=target,
        split="test",
        scaled=False,
    )

    validate_tabular_dataset(
        X_test,
        y_test,
        name=f"test_{target}",
    )

    # -------- Guardar --------
    data[target] = {
        "train": {"X": X_train, "y": y_train},
        "valid": {"X": X_valid, "y": y_valid},
        "test":  {"X": X_test,  "y": y_test},
    }


CARGANDO DATASET: delta_60

=== VALIDATION: train_delta_60 ===
n_samples_X: 54360
n_samples_y: 54360
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_index: False

=== VALIDATION: valid_delta_60 ===
n_samples_X: 11640
n_samples_y: 11640
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_index: False

=== VALIDATION: test_delta_60 ===
n_samples_X: 11700
n_samples_y: 11700
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_index: False

CARGANDO DATASET: delta_90

=== VALIDATION: train_delta_90 ===
n_samples_X: 54360
n_samples_y: 54360
n_features: 7
shape_match: True
X_has_nan: False
y_has_nan: False
X_has_inf: False
y_has_inf: False
index_equal: True
sorted_by_index: True
duplicate_index: Fal

In [53]:
X_train_delta_60 = data["delta_60"]["train"]["X"]
y_train_delta_60 = data["delta_60"]["train"]["y"]

X_valid_delta_60 = data["delta_60"]["valid"]["X"]
y_valid_delta_60 = data["delta_60"]["valid"]["y"]

X_test_delta_60  = data["delta_60"]["test"]["X"]
y_test_delta_60  = data["delta_60"]["test"]["y"]

X_train_delta_90 = data["delta_90"]["train"]["X"]
y_train_delta_90 = data["delta_90"]["train"]["y"]

X_valid_delta_90 = data["delta_90"]["valid"]["X"]
y_valid_delta_90 = data["delta_90"]["valid"]["y"]

X_test_delta_90  = data["delta_90"]["test"]["X"]
y_test_delta_90  = data["delta_90"]["test"]["y"]

### **8.3. Entrenamiento**

In [54]:
import torch

def check_gpu_available():
    print("=== GPU CHECK ===")

    cuda_available = torch.cuda.is_available()
    print("torch.cuda.is_available():", cuda_available)

    if cuda_available:
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("No hay GPU disponible")

    return cuda_available

In [55]:
USE_GPU = check_gpu_available()

=== GPU CHECK ===
torch.cuda.is_available(): False
No hay GPU disponible


In [56]:
import lightgbm as lgb
from lightgbm import LGBMRegressor


def train_evaluate_lightgbm_one2one(
    *,
    target_name: str,
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_test,
    y_test,
    n_estimators: int = 500,
    learning_rate: float = 0.05,
    num_leaves: int = 31,
    max_depth: int = -1,
    min_child_samples: int = 20,
    subsample: float = 0.8,
    colsample_bytree: float = 0.8,
    reg_alpha: float = 0.0,
    reg_lambda: float = 0.0,
    random_state: int = 42,
    n_jobs: int = -1,
    early_stopping_rounds: int = 50,
    log_evaluation_period: int = 50,
):
    """
    Entrena y evalúa un LightGBM Regressor one-to-one para un target dado.
    Usa CPU y muestra el progreso del entrenamiento.

    Retorna
    -------
    results : dict
        Contiene modelo, predicciones, métricas e información de entrenamiento.
    """
    model = LGBMRegressor(
        boosting_type="gbdt",
        objective="regression",
        metric="rmse",
        device="cpu",
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        max_depth=max_depth,
        min_child_samples=min_child_samples,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=random_state,
        n_jobs=n_jobs,
    )

    # -------------------------
    # Entrenamiento
    # -------------------------
    callbacks = [
        lgb.log_evaluation(period=log_evaluation_period),
        lgb.early_stopping(stopping_rounds=early_stopping_rounds),
    ]

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_train, y_train), (X_valid, y_valid)],
        eval_names=["train", "valid"],
        eval_metric="rmse",
        callbacks=callbacks,
    )

    # -------------------------
    # Predicciones
    # -------------------------
    y_pred_valid = model.predict(X_valid)
    y_pred_test = model.predict(X_test)

    # -------------------------
    # Métricas
    # -------------------------
    metrics_valid = evaluate_regression_predictions(
        y_true=y_valid,
        y_pred=y_pred_valid,
        split_name="valid",
        model_name="lightgbm",
        target_name=target_name,
    )

    metrics_test = evaluate_regression_predictions(
        y_true=y_test,
        y_pred=y_pred_test,
        split_name="test",
        model_name="lightgbm",
        target_name=target_name,
    )

    # -------------------------
    # Información de entrenamiento
    # -------------------------
    training_info = {
        "best_iteration": getattr(model, "best_iteration_", None),
        "best_score": getattr(model, "best_score_", None),
        "evals_result": getattr(model, "evals_result_", None),
        "device": "cpu",
    }

    results = {
        "model": model,
        "target": target_name,
        "predictions": {
            "valid": y_pred_valid,
            "test": y_pred_test,
        },
        "metrics": {
            "valid": metrics_valid,
            "test": metrics_test,
        },
        "training": training_info,
    }

    return results

In [57]:
lgbm_delta_60 = train_evaluate_lightgbm_one2one(
    target_name="delta_60",
    X_train=X_train_delta_60,
    y_train=y_train_delta_60,
    X_valid=X_valid_delta_60,
    y_valid=y_valid_delta_60,
    X_test=X_test_delta_60,
    y_test=y_test_delta_60,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=50,
    log_evaluation_period=50,
)

print_metrics(lgbm_delta_60["metrics"]["valid"])
print_metrics(lgbm_delta_60["metrics"]["test"])

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001117 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1336
[LightGBM] [Info] Number of data points in the train set: 54360, number of used features: 6
[LightGBM] [Info] Start training from score 1.395208
Training until validation scores don't improve for 50 rounds
[50]	train's rmse: 64.9635	valid's rmse: 58.7242
Early stopping, best iteration is:
[1]	train's rmse: 67.4554	valid's rmse: 58.2861
=== Regression Metrics ===
model: lightgbm
target: delta_60
split: valid
n_samples: 11640
mae: 44.851562
rmse: 58.286090
r2: -0.001999
directional_accuracy: 0.512801
=== Regression Metrics ===
model: lightgbm
target: delta_60
split: test
n_samples: 11700
mae: 77.673639
rmse: 106.570388
r2: -0.002960
directional_accuracy: 0.502906


In [58]:
lgbm_delta_90 = train_evaluate_lightgbm_one2one(
    target_name="delta_90",
    X_train=X_train_delta_90,
    y_train=y_train_delta_90,
    X_valid=X_valid_delta_90,
    y_valid=y_valid_delta_90,
    X_test=X_test_delta_90,
    y_test=y_test_delta_90,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=50,
    log_evaluation_period=50,
)

print_metrics(lgbm_delta_90["metrics"]["valid"])
print_metrics(lgbm_delta_90["metrics"]["test"])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000401 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1336
[LightGBM] [Info] Number of data points in the train set: 54360, number of used features: 6
[LightGBM] [Info] Start training from score 1.565862
Training until validation scores don't improve for 50 rounds
[50]	train's rmse: 83.7025	valid's rmse: 78.1203
Early stopping, best iteration is:
[4]	train's rmse: 86.2149	valid's rmse: 77.8007
=== Regression Metrics ===
model: lightgbm
target: delta_90
split: valid
n_samples: 11640
mae: 59.953652
rmse: 77.800738
r2: 0.000508
directional_accuracy: 0.489605
=== Regression Metrics ===
model: lightgbm
target: delta_90
split: test
n_samples: 11700
mae: 101.755772
rmse: 134.099724
r2: 0.000024
directional_accuracy: 0.522650


### **8.4. Guardado de datasets de métricas**

In [59]:
def build_and_save_one2one_metrics(
    *,
    results_delta_60: dict,
    results_delta_90: dict,
    model_name: str,
):
    """
    Construye y guarda métricas one2one para ambos targets.

    Parámetros
    ----------
    results_delta_60 : dict
    results_delta_90 : dict
    model_name : str
        Nombre del modelo (ej: 'ridge', 'random_forest')
    """

    # =========================================================
    # 1. DELTA 60
    # =========================================================
    df_valid_60 = metrics_to_df(
        results_delta_60["metrics"]["valid"]
    )

    df_test_60 = metrics_to_df(
        results_delta_60["metrics"]["test"]
    )

    df_60 = pd.concat([df_valid_60, df_test_60], ignore_index=True)

    # =========================================================
    # 2. DELTA 90
    # =========================================================
    df_valid_90 = metrics_to_df(
        results_delta_90["metrics"]["valid"]
    )

    df_test_90 = metrics_to_df(
        results_delta_90["metrics"]["test"]
    )

    df_90 = pd.concat([df_valid_90, df_test_90], ignore_index=True)

    # =========================================================
    # 3. CONSOLIDACIÓN
    # =========================================================
    df_all = pd.concat([df_60, df_90], ignore_index=True)

    # =========================================================
    # 4. GUARDADO
    # =========================================================
    save_one2one_metrics(
        df_all,
        name=f"{model_name}_all",
    )

    return df_all

In [60]:
df_lgbm = build_and_save_one2one_metrics(
    results_delta_60=lgbm_delta_60,
    results_delta_90=lgbm_delta_90,
    model_name="lgbm",
)

df_lgbm

[OK] Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/one2one_metrics/one2one_lgbm_all_metrics.parquet


,model,split,target,n_samples,mae,rmse,r2,directional_accuracy
0,lightgbm,valid,delta_60,11640,44.851562,58.286090,-0.001999,0.512801
1,lightgbm,test,delta_60,11700,77.673639,106.570388,-0.002960,0.502906
2,lightgbm,valid,delta_90,11640,59.953652,77.800738,0.000508,0.489605
3,lightgbm,test,delta_90,11700,101.755772,134.099724,0.000024,0.522650


### **PREMARKET — LightGBM**

| model    | split | target   | n_samples | mae        | rmse       | r2        | directional_accuracy |
| -------- | ----- | -------- | --------- | ---------- | ---------- | --------- | -------------------- |
| lightgbm | valid | delta_60 | 11640     | 44.851562  | 58.286090  | -0.001999 | 0.512801             |
| lightgbm | test  | delta_60 | 11700     | 77.673639  | 106.570388 | -0.002960 | 0.502906             |
| lightgbm | valid | delta_90 | 11640     | 59.953652  | 77.800738  | 0.000508  | 0.489605             |
| lightgbm | test  | delta_90 | 11700     | 101.755772 | 134.099724 | 0.000024  | 0.522650             |

---

### **OPENING — LightGBM**

| model    | split | target   | n_samples | mae       | rmse       | r2        | directional_accuracy |
| -------- | ----- | -------- | --------- | --------- | ---------- | --------- | -------------------- |
| lightgbm | valid | delta_60 | 11640     | 52.048377 | 68.792675  | 0.001021  | 0.495017             |
| lightgbm | test  | delta_60 | 11700     | 75.546386 | 102.355679 | 0.004985  | 0.494017             |
| lightgbm | valid | delta_90 | 11640     | 59.115924 | 80.097495  | 0.003465  | 0.523110             |
| lightgbm | test  | delta_90 | 11700     | 85.284998 | 113.704171 | -0.003866 | 0.540855             |

---

### **REGULAR — LightGBM**

| model    | split | target   | n_samples | mae       | rmse       | r2        | directional_accuracy |
| -------- | ----- | -------- | --------- | --------- | ---------- | --------- | -------------------- |
| lightgbm | valid | delta_60 | 46754     | 35.530062 | 48.830379  | 0.005525  | 0.545900             |
| lightgbm | test  | delta_60 | 46995     | 54.539267 | 88.469197  | -0.034147 | 0.502585             |
| lightgbm | valid | delta_90 | 46754     | 42.918212 | 60.212159  | 0.006424  | 0.548809             |
| lightgbm | test  | delta_90 | 46995     | 67.212658 | 111.704584 | -0.027837 | 0.502266             |
